[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 困难：组相对策略优化损失（GRPO）

实现 DeepSeekMath／DeepSeek-R1 中的标准**词元级组相对策略优化（GRPO）** 损失：通过同提示词组内奖励标准化替代价值网络（Critic），结合词元级 PPO 裁剪与无偏 `k3` KL 散度正则项。

### 1. 组内标准化优势
对每个提示词组 $g$，使用组内奖励均值 $\bar r_g$ 与总体标准差 $\text{std}_g = \sqrt{\frac{1}{K_g}\sum_{j \in g}(r_j - \bar r_g)^2}$（`unbiased=False`）计算标准化优势：

$$
\hat A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}
$$

### 2. 词元级裁剪代理目标与无偏 `k3` KL 惩罚
对每个词元位置 $(i, t)$，定义重要性采样比率 $r_{i,t} = \exp(\text{new\_logps}_{i,t} - \text{old\_logps}_{i,t})$，构造对称裁剪目标：

$$
s_{i,t} = \min\Big(r_{i,t}\hat A_i,\; \operatorname{clip}(r_{i,t},\, 1 - \varepsilon,\, 1 + \varepsilon)\,\hat A_i\Big)
$$

当提供参考策略对数概率 `ref_logps` 且 `kl_coef`（$\beta$）$> 0$ 时，使用 Schulman 无偏非负 `k3` 估计器计算每词元 KL 散度（令 $\Delta_{i,t} = \text{ref\_logps}_{i,t} - \text{new\_logps}_{i,t}$）：

$$
\mathbb{D}_{\text{KL}, i, t} = \exp(\Delta_{i,t}) - \Delta_{i,t} - 1
$$

### 3. 先序列内按长度平均，再按批次平均
记第 $i$ 条回复的有效词元数为 $|o_i| = \sum_{t=1}^{T} m_{i,t}$（未提供 `mask` 时 $m_{i,t} = 1, |o_i| = T$），标准 GRPO 先对每条回复除以自身长度 $|o_i|$，再对批次 $B$ 求均值：

$$
\mathcal{L}_{\text{GRPO}} = -\frac{1}{B} \sum_{i=1}^{B} \frac{1}{|o_i|} \sum_{t=1}^{T} m_{i,t} \Big(s_{i,t} - \beta\,\mathbb{D}_{\text{KL}, i, t}\Big)
$$

实现注意事项：将 `old_logps`、`ref_logps` 和优势 $\hat A_i$ 从计算图中分离（注意 $\mathbb{D}_{\text{KL}, i, t}$ 中的 `new_logps` 需保留梯度），使梯度仅通过 `new_logps` 传播。

### 函数或类接口
```python
from torch import Tensor

def grpo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              ref_logps: Tensor | None = None,
              clip_ratio: float = 0.2,
              kl_coef: float = 0.0,
              eps: float = 1e-6) -> Tensor:
    """计算标准词元级 GRPO 损失。

    new_logps: (B, T) 当前策略的词元级对数概率
    old_logps: (B, T) 旧策略（采样策略）的词元级对数概率
    rewards: (B,) 每条采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    ref_logps: 可选，形状为 (B, T) 的参考策略词元级对数概率
    clip_ratio: PPO 对称裁剪范围 eps
    kl_coef: KL 正则项系数 beta
    eps: 组内标准差归一化分母中的稳定项
    返回：标量损失（Tensor）
    """
```

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✏️ 在此编写你的实现

from torch import Tensor

def grpo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              ref_logps: Tensor | None = None,
              clip_ratio: float = 0.2,
              kl_coef: float = 0.0,
              eps: float = 1e-6) -> Tensor:
    pass  # 组内 z-score 优势 + 词元级对称裁剪 + 可选 k3 KL + 每序列按 |o_i| 平均后再按 B 平均

In [ ]:
# 🧪 调试
ratios = torch.tensor([[1.30, 1.10], [0.70, 0.90]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', grpo_loss(new_logps, old_logps, rewards, group_ids, eps=0.0).item())

In [ ]:
# ✅ 提交
from torch_judge import check
check('grpo_loss')